# 第 1 周 第 2 天练习 —— 把 Day 1 升级为本地 Ollama

## 练习目标（理念）

在 Day 1「抓网页 → 吐槽式摘要」流水线不变的前提下，把云端 OpenRouter 换成 **本地 Ollama**：

- **同一套** system / user prompt 与 `messages_for` / `summarize` / `display_summary`
- **换底座**：`OpenAI(base_url=http://localhost:11434/v1, api_key='ollama')`
- **换模型**：`llama3.2:1b`（需本机已 `ollama pull`）

用来对比：同一提示词下，云端小模型 vs 本地小模型的语气与质量差异。

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama OpenAI 兼容接口 | `http://localhost:11434/v1` |
| 本地模型名 | `llama3.2:1b` |
| 复用 Day 1 抓取与 messages | `fetch_website_contents` + `messages_for` |
| 无需云端 Key | `api_key='ollama'` 仅占位 |

## 怎么跑

1. 启动 Ollama，并确保已拉取 `llama3.2:1b`
2. 同目录需有 `scraper.py`
3. 从上到下运行；最后一格仍摘要 `http://juunsdev.my.id`


### 导入依赖

引入环境变量、`dotenv`、本地抓取器、IPython 展示与 OpenAI 客户端——后面用同一 SDK 打 Ollama 兼容口。


In [ ]:
# ========== 导入：Day 2 仍用同一套工具箱，只是后端改成本地 ==========

# 导入标准库 os：本格虽未立刻用到，常与 dotenv / 环境配置一起保留
import os
# 从 dotenv 导入 load_dotenv：本笔记本后续格未显式调用，但保留导入与 Day 1 结构一致
from dotenv import load_dotenv
# 从同目录 scraper 导入抓取函数：URL → 网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：用 Markdown 渲染模型回复
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：对接 Ollama 的 /v1 兼容 API
from openai import OpenAI


### 环境准备：指向本地 Ollama 的 OpenAI 兼容客户端

`base_url` 固定为本机 `11434/v1`；`api_key` 用占位字符串即可（Ollama 通常不校验）。


In [ ]:
# ========== Ollama 客户端：OpenAI SDK + 本地 base_url ==========

# Ollama 对外暴露的 OpenAI 兼容根地址（端口 11434 是默认）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建客户端；api_key='ollama' 仅为占位，本地服务一般不检查密钥
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


### 环境准备：messages、摘要与展示（与 Day 1 同构）

提示词与函数结构几乎照搬 Day 1；唯一实质差异是 `chat.completions.create` 打到 `ollama` 并指定本地模型名。


In [ ]:
# ========== Prompt + 业务函数：逻辑同 Day 1，客户端换成 ollama ==========

# system prompt 保留英文：吐槽式、短摘要、忽略导航、直接回 Markdown（勿包代码块）
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
# user 前缀保留英文：说明正文来源，并要求顺带总结新闻/公告
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# 组装 Chat Completions 的 messages：system 定调 + user 放正文
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

# 端到端摘要：抓网页 → 本地 llama3.2:1b → 返回助手文本
def summarize(url):
  # 与 Day 1 相同：先把 URL 变成纯文本
  website = fetch_website_contents(url)
  # 关键：客户端是 ollama，模型是本地 tag llama3.2:1b（需已 pull）
  response = ollama.chat.completions.create(
      model = "llama3.2:1b",
      messages = messages_for(website)
  )
  # 取出第一条 choice 的 message.content
  return response.choices[0].message.content

# 笔记本展示封装：summarize 后用 Markdown 渲染
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


### 抓取并摘要自己的网站（本地模型）

与 Day 1 同一 URL，便于并排对比 OpenRouter 与 Ollama 的摘要风格。


In [ ]:
# ========== 实测：对个人站跑本地 Ollama 摘要 ==========

# URL 保持原样；可改成其他站点做对比实验
display_summary("http://juunsdev.my.id")
